# Lecture 20: Regression Trees

**Case:** Predict the price of a used Toyota Camry from its age and features.

The sequence follows Gábor's Chapter 15 CART case: begin with age alone, read a
small tree, compare it with a linear model, expose overfitting, then tune a
multiple-feature tree and evaluate it on untouched data.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

sns.set_theme(style="whitegrid")

In [ ]:
cars = pd.read_csv("../../data/raw/used_cars_course.csv")
cars[["price", "age", "odometer_10k_miles"]].describe().T.round(1)

## 1. Compare a linear relationship with a shallow tree

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=cars, x="age", y="price", lowess=True, scatter_kws={"alpha": 0.55}, ax=ax)
ax.set(title="Used-car price and age", ylabel="Price (USD)")
plt.show()

In [ ]:
train, test = train_test_split(cars, test_size=0.25, random_state=42)
age_tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20, random_state=42)
age_tree.fit(train[["age"]], train["price"])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(age_tree, feature_names=["age"], filled=True, rounded=True, ax=ax)
ax.set_title("A depth-two regression tree")
plt.show()

In [ ]:
example_ages = pd.DataFrame({"age": [2, 5, 8, 12, 16]})
example_ages["tree_prediction"] = age_tree.predict(example_ages)
example_ages.round(0)

In [ ]:
age_linear = LinearRegression().fit(train[["age"]], train["price"])
age_comparison = pd.Series(
    {
        "linear_test_rmse": root_mean_squared_error(test["price"], age_linear.predict(test[["age"]])),
        "tree_test_rmse": root_mean_squared_error(test["price"], age_tree.predict(test[["age"]])),
    }
)
age_comparison.round(0)

## 2. Add features through a reproducible pipeline

In [ ]:
target = "price"
numeric = ["age", "odometer_10k_miles", "dealer"]
categorical = ["condition", "cylinders", "drive", "area", "trim"]
features = numeric + categorical
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

In [ ]:
preprocess = ColumnTransformer(
    [
        ("numeric", "passthrough", numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ]
)
unrestricted = Pipeline(
    [("preprocess", preprocess), ("model", DecisionTreeRegressor(random_state=42))]
)
unrestricted.fit(X_train, y_train)

In [ ]:
pd.Series(
    {
        "training_rmse": root_mean_squared_error(y_train, unrestricted.predict(X_train)),
        "test_rmse": root_mean_squared_error(y_test, unrestricted.predict(X_test)),
    }
).round(0)

A very small training error paired with a much larger test error is evidence of
overfitting. We restrict the tree using hyperparameters chosen by
cross-validation—not by looking repeatedly at the test set.

## 3. Tune complexity on the training data

In [ ]:
tree_pipeline = Pipeline(
    [("preprocess", preprocess), ("model", DecisionTreeRegressor(random_state=42))]
)
parameter_grid = {
    "model__max_depth": [2, 3, 4, 5, 7, None],
    "model__min_samples_leaf": [5, 10, 20, 35],
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    tree_pipeline, parameter_grid, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1,
)
search.fit(X_train, y_train)

In [ ]:
results = pd.DataFrame(search.cv_results_)
results[["param_model__max_depth", "param_model__min_samples_leaf", "mean_test_score"]].assign(
    cv_rmse=lambda frame: -frame["mean_test_score"]
).nsmallest(8, "cv_rmse").round(1)

In [ ]:
best_tree = search.best_estimator_
test_prediction = best_tree.predict(X_test)
print(search.best_params_)
print(f"Cross-validated training RMSE: {-search.best_score_:.0f}")
print(f"Untouched test RMSE: {root_mean_squared_error(y_test, test_prediction):.0f}")

## 4. Interpret the selected tree cautiously

In [ ]:
transformed_names = best_tree.named_steps["preprocess"].get_feature_names_out()
fitted_tree = best_tree.named_steps["model"]
importance = pd.Series(fitted_tree.feature_importances_, index=transformed_names)
importance.sort_values(ascending=False).head(12).round(3)

In [ ]:
permutation = permutation_importance(
    best_tree, X_test, y_test, n_repeats=20, random_state=42,
    scoring="neg_root_mean_squared_error",
)
pd.Series(permutation.importances_mean, index=features).sort_values(ascending=False).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    fitted_tree, feature_names=transformed_names,
    filled=True, rounded=True, max_depth=3, ax=ax,
)
ax.set_title("Top levels of the tuned regression tree")
plt.show()

Feature importance describes the fitted prediction rule, not a causal effect.
Correlated features can substitute for one another, and impurity-based
importance can favor variables offering many possible split points.

## Session 6 reconstruction handoff

Open `prompts/session06-tree-reconstruction.md`. Audit the generated split
logic, stopping criteria, validation design, metric, feature-importance claims,
and use of the test set.

### Optional extension: cost-complexity pruning

Obtain the pruning path from the training data, select several candidate
`ccp_alpha` values, and compare them by cross-validation. This is post-pruning;
`max_depth` and `min_samples_leaf` are pre-pruning controls.

In [ ]:
transformed_train = best_tree.named_steps["preprocess"].transform(X_train)
pruning_path = DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(
    transformed_train, y_train
)
pd.Series(pruning_path.ccp_alphas).describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(3)